# SchemaAware-NL2SQL: Demo Notebook

This notebook demonstrates the core capability of SchemaAware-NL2SQL:
generating accurate SQL from natural language using **only the database schema**.

No sample data. No hardcoded domain knowledge. No fine-tuning required.

The same model is applied to three completely different domains:
1. HR / Employee Management
2. Retail / E-Commerce
3. Healthcare

---
**Prerequisites:** Set your `OPENAI_API_KEY` in the environment or in the cell below.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

# Set your API key here if not in environment
# os.environ['OPENAI_API_KEY'] = 'sk-...'

from src.sql_generator import SchemaAwareNL2SQL
from src.utils import format_schema_table, validate_sql_syntax

print('SchemaAware-NL2SQL loaded successfully.')

## Domain 1: HR / Employee Management

Standard enterprise HR schema with employees and departments.

In [ ]:
hr_schema = {
    'employees': {
        'columns': ['employee_id', 'name', 'department_id', 'salary', 'hire_date', 'job_title'],
        'types':   ['INT', 'VARCHAR', 'INT', 'DECIMAL', 'DATE', 'VARCHAR'],
        'pk': 'employee_id'
    },
    'departments': {
        'columns': ['department_id', 'department_name', 'location', 'budget'],
        'types':   ['INT', 'VARCHAR', 'VARCHAR', 'DECIMAL'],
        'pk': 'department_id'
    },
    'performance_reviews': {
        'columns': ['review_id', 'employee_id', 'review_year', 'score', 'reviewer_id'],
        'types':   ['INT', 'INT', 'INT', 'DECIMAL', 'INT'],
        'pk': 'review_id'
    }
}

hr_fk = [
    ('employees.department_id', 'departments.department_id'),
    ('performance_reviews.employee_id', 'employees.employee_id')
]

hr_model = SchemaAwareNL2SQL(schema=hr_schema, foreign_keys=hr_fk)

print('Schema loaded:')
print(format_schema_table(hr_model.schema_context))

In [ ]:
questions = [
    'List the names and salaries of all employees in the Engineering department hired after 2020',
    'Which department has the highest average salary?',
    'Show the top 3 employees by performance score in 2023',
]

for q in questions:
    sql = hr_model.generate(q)
    valid, _ = validate_sql_syntax(sql)
    print(f'Q: {q}')
    print(f'SQL:\n{sql}')
    print(f'Valid: {valid}\n{"="*60}')

## Domain 2: Retail / E-Commerce

**Zero changes to the model.** We load a completely different schema and generate SQL.

In [ ]:
retail_schema = {
    'customers': {
        'columns': ['customer_id', 'name', 'email', 'region', 'tier'],
        'types':   ['INT', 'VARCHAR', 'VARCHAR', 'VARCHAR', 'VARCHAR'],
        'pk': 'customer_id'
    },
    'orders': {
        'columns': ['order_id', 'customer_id', 'order_date', 'status', 'total_amount'],
        'types':   ['INT', 'INT', 'DATE', 'VARCHAR', 'DECIMAL'],
        'pk': 'order_id'
    },
    'order_items': {
        'columns': ['item_id', 'order_id', 'product_id', 'quantity', 'unit_price'],
        'types':   ['INT', 'INT', 'INT', 'INT', 'DECIMAL'],
        'pk': 'item_id'
    },
    'products': {
        'columns': ['product_id', 'name', 'category', 'price', 'stock_qty'],
        'types':   ['INT', 'VARCHAR', 'VARCHAR', 'DECIMAL', 'INT'],
        'pk': 'product_id'
    }
}

retail_fk = [
    ('orders.customer_id',       'customers.customer_id'),
    ('order_items.order_id',     'orders.order_id'),
    ('order_items.product_id',   'products.product_id'),
]

# Same model class — just a new schema
retail_model = SchemaAwareNL2SQL(schema=retail_schema, foreign_keys=retail_fk)

questions = [
    'Show total revenue by region for Q1 2024',
    'List the top 5 best-selling products by quantity sold',
    'Which premium customers placed more than 3 orders in the last 90 days?',
]

for q in questions:
    sql = retail_model.generate(q)
    print(f'Q: {q}')
    print(f'SQL:\n{sql}\n{"="*60}')

## Domain 3: Healthcare

A completely different domain. Same model, same code, different schema.

In [ ]:
healthcare_ddl = """
    CREATE TABLE patients (
        patient_id INT PRIMARY KEY,
        name VARCHAR(100),
        dob DATE,
        gender VARCHAR(10),
        insurance_id INT
    );
    CREATE TABLE appointments (
        appointment_id INT PRIMARY KEY,
        patient_id INT,
        doctor_id INT,
        appointment_date DATE,
        diagnosis_code VARCHAR(20),
        FOREIGN KEY (patient_id) REFERENCES patients(patient_id)
    );
    CREATE TABLE doctors (
        doctor_id INT PRIMARY KEY,
        name VARCHAR(100),
        specialty VARCHAR(100),
        department VARCHAR(100)
    );
"""

# DDL input — foreign keys parsed automatically
health_model = SchemaAwareNL2SQL(schema=healthcare_ddl)

questions = [
    'How many appointments did each doctor have in 2024?',
    'List patients who were seen by a cardiologist more than twice',
]

for q in questions:
    sql = health_model.generate(q)
    print(f'Q: {q}')
    print(f'SQL:\n{sql}\n{"="*60}')

## Cross-Domain Switching on a Single Instance

`update_schema()` lets you swap the schema without creating a new instance —
useful in multi-tenant deployments where the schema changes per customer.

In [ ]:
# Start with HR schema
model = SchemaAwareNL2SQL(schema=hr_schema, foreign_keys=hr_fk)
print('HR result:')
print(model.generate('How many employees joined this year?'))

# Switch to retail — no new instance
model.update_schema(retail_schema, retail_fk)
print('\nRetail result (same model instance):')
print(model.generate('How many orders were placed this year?'))